# IT Service Ticket Classification — Fine-Tuning and Lightweight Optimization

## 1. Introduction

This notebook focuses on fine-tuning a pre-trained DistilBERT model for IT ticket classification. The objective is to evaluate different training configurations to determine the optimal setup for model performance.

Multiple experiments were conducted by varying the number of training epochs while keeping other hyperparameters constant. The goal is to identify the best-performing model based on macro F1-score and generalization performance.


## 2. Import Libraries

In [1]:
# Import os to work with directories and file paths.
import os

# Import json to load label mappings from JSON files.
import json

# Import numpy to convert model logits into predicted class IDs.
import numpy as np

# Import pandas to organize and analyze experiment results.
import pandas as pd

# Import torch to enable model training and check GPU availability.
import torch

# Import load_from_disk to load the preprocessed Hugging Face DatasetDict from disk.
from datasets import load_from_disk

# Import evaluate to compute model evaluation metrics.
import evaluate

# Import AutoTokenizer to load the tokenizer associated with the pretrained model.
from transformers import AutoTokenizer

# Import AutoModelForSequenceClassification to load the pretrained model with a classification head.
from transformers import AutoModelForSequenceClassification

# Import TrainingArguments to define the configuration for each training experiment.
from transformers import TrainingArguments

# Import Trainer to manage the fine-tuning and evaluation process.
from transformers import Trainer

# Import DataCollatorWithPadding to dynamically pad input sequences during batching.
from transformers import DataCollatorWithPadding

# Import classification_report to evaluate performance at the class level.
from sklearn.metrics import classification_report

# Import confusion_matrix to analyze prediction errors across classes.
from sklearn.metrics import confusion_matrix

## 3. Define Project Paths

In [2]:
# Define the path to the tokenized Hugging Face DatasetDict stored in the processed data directory.
dataset_path = "../data/tokenized/tickets_distilbert"

# Define the path to the label-to-ID mapping JSON file.
label2id_path = "../data/metadata/label2id.json"

# Define the path to the ID-to-label mapping JSON file.
id2label_path = "../data/metadata/id2label.json"

# Define the base directory where all fine-tuned models will be saved.
models_base_path = "../models"

# Define the pretrained model checkpoint used for transfer learning.
model_checkpoint = "distilbert-base-uncased"

## 4. Load Tokenized Dataset and Label Mappings

In [3]:
# Load the tokenized Hugging Face DatasetDict from disk.
dataset = load_from_disk(dataset_path)

# Display the dataset structure to confirm that train, validation, and test splits are available.
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 27024
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5791
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5792
    })
})

In [4]:
# Open the label-to-ID mapping JSON file.
with open(label2id_path, "r") as file:
    # Load the label-to-ID mapping into a Python dictionary.
    label2id = json.load(file)

# Open the ID-to-label mapping JSON file.
with open(id2label_path, "r") as file:
    # Load the ID-to-label mapping into a Python dictionary.
    id2label = json.load(file)

# Convert JSON keys from strings to integers, as required by the model configuration.
id2label = {int(key): value for key, value in id2label.items()}

# Determine the number of target classes.
num_labels = len(label2id)

# Display the number of classes for verification.
print("Number of labels:", num_labels)

# Display the label-to-ID mapping for inspection.
label2id

Number of labels: 6


{'Hardware': 0,
 'HR Support': 1,
 'Access': 2,
 'Storage': 3,
 'Purchase': 4,
 'Administrative rights': 5}

In [16]:
# Display the number of examples in the training split.
print("Train examples:", len(dataset["train"]))

# Display the number of examples in the validation split.
print("Validation examples:", len(dataset["validation"]))

# Display the number of examples in the test split.
print("Test examples:", len(dataset["test"]))

Train examples: 27024
Validation examples: 5791
Test examples: 5792


## 5. Load Tokenizer and Configure Device

In [17]:
# Load the tokenizer associated with the selected checkpoint.
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Create a data collator that dynamically pads each batch during training.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Check whether a GPU is available for faster training.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Display the device that will be used by the training process.
print("Training device:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Training device: cuda


## 6. Define Evaluation Metrics

In [18]:
# Load the accuracy metric from the Evaluate library.
accuracy_metric = evaluate.load("accuracy")

# Load the F1-score metric from the Evaluate library.
f1_metric = evaluate.load("f1")

# Define a function that calculates evaluation metrics during validation.
def compute_metrics(eval_pred):
    # Unpack the model outputs and true labels from the evaluation prediction object.
    logits, labels = eval_pred

    # Convert logits into predicted class IDs by selecting the class with the highest score.
    predictions = np.argmax(logits, axis=-1)

    # Calculate accuracy using the predicted labels and true labels.
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]

    # Calculate macro F1-score to evaluate performance across all classes equally.
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]

    # Return the metrics to the Trainer.
    return {"accuracy": accuracy, "f1_macro": f1_macro}

## 7. Define Experiment Configurations

In [ ]:
# Define the list of fine-tuning experiments to run.
experiments = [
    # Baseline configuration used as the reference model.
    {"run_name": "run_01_baseline_2_epochs", "model_dir": "distilbert_ticket_classifier_v1", "epochs": 2, "batch_size": 16, "learning_rate": 2e-5},

    # Define a shorter training configuration to compare whether one epoch is enough.
    {"run_name": "run_02_short_1_epoch", "model_dir": "distilbert_ticket_classifier_v2", "epochs": 1, "batch_size": 16, "learning_rate": 2e-5},

    # Define a longer training configuration to test whether one additional epoch improves results.
    {"run_name": "run_03_long_3_epochs", "model_dir": "distilbert_ticket_classifier_v3", "epochs": 3, "batch_size": 16, "learning_rate": 2e-5},
]

# Display the experiment plan before training.
pd.DataFrame(experiments)

,run_name,model_dir,epochs,batch_size,learning_rate
0,run_01_baseline_2_epochs,distilbert_ticket_classifier_v1,2,16,0.00002
1,run_02_short_1_epoch,distilbert_ticket_classifier_v2,1,16,0.00002
2,run_03_long_3_epochs,distilbert_ticket_classifier_v3,3,16,0.00002


## 8. Create Reusable Training Function

In [20]:
# Define a function to create a fresh DistilBERT classification model for each experiment.
def load_fresh_model():
    # Load DistilBERT with a sequence classification head for the ticket categories.
    model = AutoModelForSequenceClassification.from_pretrained(
        # Use the pretrained DistilBERT checkpoint as the base model.
        model_checkpoint,
        # Set the number of target labels.
        num_labels=num_labels,
        # Attach the ID-to-label mapping to the model configuration.
        id2label=id2label,
        # Attach the label-to-ID mapping to the model configuration.
        label2id=label2id,
    )

    # Move the model to the selected training device.
    model.to(device)

    # Return the initialized model.
    return model

# Define a function that trains, evaluates, and saves one experiment.
def run_experiment(experiment):
    # Store the experiment run name for logging and folder naming.
    run_name = experiment["run_name"]

    # Build the output path where this experiment model will be saved.
    model_output_path = os.path.join(models_base_path, experiment["model_dir"])

    # Create a fresh model so each experiment starts from the same pretrained checkpoint.
    model = load_fresh_model()

    # Define the training configuration for this experiment.
    training_args = TrainingArguments(
        # Save training outputs in the model-specific output folder.
        output_dir=model_output_path,
        # Store the readable name of this experiment.
        run_name=run_name,
        # Set the learning rate for fine-tuning.
        learning_rate=experiment["learning_rate"],
        # Set the batch size used by each device during training.
        per_device_train_batch_size=experiment["batch_size"],
        # Set the batch size used by each device during evaluation.
        per_device_eval_batch_size=experiment["batch_size"],
        # Set the number of training epochs for this experiment.
        num_train_epochs=experiment["epochs"],
        # Evaluate the model at the end of each epoch.
        eval_strategy="epoch",
        # Disable automatic checkpoint saving to reduce disk usage.
        save_strategy="no",
        # Log training metrics at the end of each epoch.
        logging_strategy="epoch",
        # Disable external reporting integrations for this notebook.
        report_to="none",
        # Use mixed precision when CUDA is available to improve GPU training speed.
        fp16=torch.cuda.is_available(),
    )

    # Initialize the Hugging Face Trainer for this experiment.
    trainer = Trainer(
        # Pass the fresh model that will be fine-tuned.
        model=model,
        # Pass the training configuration.
        args=training_args,
        # Provide the tokenized training dataset.
        train_dataset=dataset["train"],
        # Provide the tokenized validation dataset.
        eval_dataset=dataset["validation"],
        # Provide the data collator for batch preparation.
        data_collator=data_collator,
        # Provide the metric function for validation evaluation.
        compute_metrics=compute_metrics,
    )

    # Print which experiment is starting.
    print(f"Starting experiment: {run_name}")

    # Start the fine-tuning process.
    trainer.train()

    # Evaluate the trained model on the validation split.
    validation_results = trainer.evaluate(eval_dataset=dataset["validation"])

    # Evaluate the trained model on the test split.
    test_results = trainer.evaluate(eval_dataset=dataset["test"])

    # Create the model output folder if it does not already exist.
    os.makedirs(model_output_path, exist_ok=True)

    # Save the fine-tuned model to disk.
    trainer.save_model(model_output_path)

    # Save the tokenizer with the model to make future inference easier.
    tokenizer.save_pretrained(model_output_path)

    # Store the main results in a compact dictionary.
    result = {
        # Store the run name.
        "run_name": run_name,
        # Store the model folder path.
        "model_path": model_output_path,
        # Store the number of epochs used.
        "epochs": experiment["epochs"],
        # Store the batch size used.
        "batch_size": experiment["batch_size"],
        # Store the learning rate used.
        "learning_rate": experiment["learning_rate"],
        # Store validation accuracy.
        "validation_accuracy": validation_results.get("eval_accuracy"),
        # Store validation macro F1-score.
        "validation_f1_macro": validation_results.get("eval_f1_macro"),
        # Store test accuracy.
        "test_accuracy": test_results.get("eval_accuracy"),
        # Store test macro F1-score.
        "test_f1_macro": test_results.get("eval_f1_macro"),
    }

    # Print where the model was saved.
    print("Model saved to:", model_output_path)

    # Return the trainer and result for later analysis.
    return trainer, result

## 9. Run Fine-Tuning Experiments

In [21]:
# Create an empty list to store experiment results.
experiment_results = []

# Create an empty dictionary to keep each trained Trainer object available in memory.
trained_trainers = {}

# Loop through each experiment configuration.
for experiment in experiments:
    # Run the current fine-tuning experiment.
    trainer, result = run_experiment(experiment)

    # Store the trained Trainer object using the run name as the key.
    trained_trainers[experiment["run_name"]] = trainer

    # Append the compact metrics dictionary to the results list.
    experiment_results.append(result)

# Convert all experiment results into a DataFrame for comparison.
results_df = pd.DataFrame(experiment_results)

# Display the experiment comparison table.
results_df

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting experiment: run_01_baseline_2_epochs


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.509379,0.369461,0.881195,0.876257
2,0.250474,0.319681,0.898463,0.889074


...

Starting experiment: run_03_long_3_epochs


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.497128,0.383098,0.872906,0.867652
2,0.250427,0.322513,0.900708,0.893633
3,0.162796,0.338586,0.903644,0.895541


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ../models/distilbert_ticket_classifier_v3


,run_name,model_path,epochs,batch_size,learning_rate,validation_accuracy,validation_f1_macro,test_accuracy,test_f1_macro
0,run_01_baseline_2_epochs,../models/distilbert_ticket_classifier_v1,2,16,0.00002,0.898463,0.889074,0.897445,0.891165
1,run_02_short_1_epoch,../models/distilbert_ticket_classifier_v2,1,16,0.00002,0.885857,0.875770,0.881043,0.871523
2,run_03_long_3_epochs,../models/distilbert_ticket_classifier_v3,3,16,0.00002,0.903644,0.895541,0.902624,0.896348


## 10. Select Best Candidate by Validation Macro F1

In [27]:
# Select the most relevant columns for model comparison.
comparison_df = results_df[
    [
        "run_name",
        "epochs",
        "batch_size",
        "learning_rate",
        "validation_accuracy",
        "validation_f1_macro",
        "test_accuracy",
        "test_f1_macro"
    ]
]

# Display the final experiment comparison table.
comparison_df

,run_name,epochs,batch_size,learning_rate,validation_accuracy,validation_f1_macro,test_accuracy,test_f1_macro
0,run_01_baseline_2_epochs,2,16,0.00002,0.898463,0.889074,0.897445,0.891165
1,run_02_short_1_epoch,1,16,0.00002,0.885857,0.875770,0.881043,0.871523
2,run_03_long_3_epochs,3,16,0.00002,0.903644,0.895541,0.902624,0.896348


### Model Selection Criteria

The best model was selected based on macro F1-score, as it provides a balanced evaluation across all classes. Special attention was given to consistency between validation and test performance to ensure generalization.

Among the evaluated configurations, the 3-epoch model achieved the highest macro F1-score on both validation (0.8955) and test (0.8963) sets, with consistent performance across splits.

This model was selected as the final candidate, as it provides the best trade-off between performance improvement and training cost.

In [22]:
# Identify the row with the highest validation macro F1-score.
best_row = results_df.loc[results_df["validation_f1_macro"].idxmax()]

# Extract the best run name from the selected row.
best_run_name = best_row["run_name"]

# Extract the best model path from the selected row.
best_model_path = best_row["model_path"]

# Display the best run metadata without adding conclusions.
best_row

,2
run_name,run_03_long_3_epochs
model_path,../models/distilbert_ticket_classifier_v3
epochs,3
batch_size,16
learning_rate,0.00002
validation_accuracy,0.903644
validation_f1_macro,0.895541
test_accuracy,0.902624
test_f1_macro,0.896348


### Best Model Identification

The best-performing model was identified by selecting the configuration with the highest validation macro F1-score. This approach ensures that the selected model achieves a balanced performance across all classes, rather than being biased toward the majority class.

The selected model corresponds to the 3-epoch configuration (run_03_long_3_epochs), which achieved:

- Validation F1-score: 0.8955  
- Test F1-score: 0.8963  
- Test Accuracy: 0.9026  

These results confirm that the model not only performs well on the validation set but also generalizes effectively to unseen data.

The consistency between validation and test metrics indicates that the model does not exhibit overfitting, making it a strong candidate for downstream tasks such as deployment and inference.

## 11. Generate Test Classification Report for the Best Candidate

In [23]:
# Retrieve the Trainer object for the best run.
best_trainer = trained_trainers[best_run_name]

# Generate predictions on the test split using the best candidate model.
test_predictions_output = best_trainer.predict(dataset["test"])

# Convert model logits into predicted class IDs.
test_predictions = np.argmax(test_predictions_output.predictions, axis=-1)

# Extract the true labels from the prediction output.
test_labels = test_predictions_output.label_ids

# Create the ordered list of class names using the ID-to-label mapping.
target_names = [id2label[index] for index in sorted(id2label.keys())]

# Generate the test classification report as a dictionary.
report_dict = classification_report(test_labels, test_predictions, target_names=target_names, output_dict=True)

# Convert the classification report into a DataFrame for easier inspection.
classification_report_df = pd.DataFrame(report_dict).transpose()

# Display the classification report.
classification_report_df

,precision,recall,f1-score,support
Hardware,0.893119,0.897059,0.895084,2040.000000
HR Support,0.904501,0.908924,0.906707,1636.000000
Access,0.908088,0.925961,0.916937,1067.000000
Storage,0.911765,0.894231,0.902913,416.000000
Purchase,0.966574,0.940379,0.953297,369.000000
Administrative rights,0.836066,0.772727,0.803150,264.000000
accuracy,0.902624,0.902624,0.902624,0.902624
macro avg,0.903352,0.889880,0.896348,5792.000000
weighted avg,0.902510,0.902624,0.902474,5792.000000


### Final Model Performance Analysis

The selected model was evaluated in detail using a classification report on the test set to understand its performance at the class level.

Overall, the model achieved:

- Accuracy: 0.9026  
- Macro F1-score: 0.8963  

These results confirm strong overall performance and balanced classification across categories.

Class-Level Insights

Most classes show high precision and recall, indicating that the model is able to correctly identify and distinguish between different ticket categories:

- "Access" and "Purchase" exhibit the highest performance, with F1-scores above 0.91 and 0.95 respectively, suggesting that these categories have clear and distinguishable patterns in the data.
- "Hardware", "HR Support", and "Storage" also perform consistently well, with balanced precision and recall values.

However, the "Administrative rights" class shows lower recall (≈ 0.77) and F1-score (≈ 0.80), indicating that the model has more difficulty correctly identifying this category. This may be due to:

- Lower representation in the dataset (class imbalance)
- Overlapping patterns with other categories
- Less distinctive language features

Interpretation

Despite the lower performance in one class, the overall macro F1-score remains high, demonstrating that the model performs well across all categories on average.

The confusion matrix further supports this observation, showing that most misclassifications are limited and occur between semantically related categories.

## 12. Generate Test Confusion Matrix for the Best Candidate

In [24]:
# Calculate the confusion matrix using true labels and predicted labels.
cm = confusion_matrix(test_labels, test_predictions)

# Convert the confusion matrix into a DataFrame with class names as rows and columns.
confusion_matrix_df = pd.DataFrame(cm, index=target_names, columns=target_names)

# Display the confusion matrix as a table.
confusion_matrix_df

,Hardware,HR Support,Access,Storage,Purchase,Administrative rights
Hardware,1830,107,56,16,6,25
HR Support,99,1487,34,10,4,2
Access,45,26,988,3,2,3
Storage,16,14,7,372,0,7
Purchase,13,4,2,0,347,3
Administrative rights,46,6,1,7,0,204


### Confusion Matrix Analysis

The confusion matrix provides a detailed view of how the model performs across individual classes by showing correct predictions and misclassifications.

Most predictions are concentrated along the diagonal, indicating that the model correctly classifies the majority of samples across all categories.

Key Observations

- "Hardware" and "HR Support" show strong performance, with a high number of correct classifications (1830 and 1487 respectively), although some confusion exists between these two classes.
- "Access" and "Purchase" exhibit very high precision, with minimal misclassifications, suggesting that these categories are well-defined and easily distinguishable.
- "Storage" also performs well, with only a small number of misclassified samples.
- The "Administrative rights" class shows the highest level of confusion, particularly being misclassified as "Hardware" and, to a lesser extent, other categories.

Interpretation

The observed misclassifications suggest that some classes share similar semantic patterns, which makes them harder to distinguish. In particular:

- Confusion between "Hardware" and "HR Support" may indicate overlapping terminology or similar ticket descriptions.
- Lower performance for "Administrative rights" is consistent with the classification report, reinforcing the idea that this class is more challenging due to either limited data or less distinctive features.

## 13. Optional: Compress Saved Models for Download

In [25]:
# Import shutil to compress model folders into zip files.
import shutil

# Loop through each experiment configuration.
for experiment in experiments:
    # Build the folder path of the saved model.
    model_folder = os.path.join(models_base_path, experiment["model_dir"])

    # Build the zip output path without the .zip extension.
    zip_base_path = os.path.join(models_base_path, experiment["model_dir"])

    # Compress the saved model folder into a zip file.
    shutil.make_archive(zip_base_path, "zip", model_folder)

    # Print the generated zip file path.
    print("Created:", zip_base_path + ".zip")

Created: ../models/distilbert_ticket_classifier_v1.zip
Created: ../models/distilbert_ticket_classifier_v2.zip
Created: ../models/distilbert_ticket_classifier_v3.zip


## 14. Conclusion

In this stage, multiple fine-tuning configurations of a DistilBERT model were evaluated to identify the optimal training setup for the IT ticket classification task.

Three experiments were conducted using 1, 2, and 3 training epochs while keeping the remaining hyperparameters constant (batch size = 16, learning rate = 2e-5). The results showed a clear and consistent performance trend:

- The 1-epoch model achieved a test macro F1-score of 0.8715, indicating underfitting and insufficient learning from the dataset.
- The 2-epoch model improved performance significantly, reaching a test macro F1-score of 0.8912, establishing a strong baseline.
- The 3-epoch model achieved the best overall performance, with a test macro F1-score of 0.8963 and test accuracy of 0.9026.

Importantly, the 3-epoch configuration maintained consistent performance between validation (F1 = 0.8955) and test (F1 = 0.8963) sets, indicating good generalization and no signs of overfitting.

Based on these results, the 3-epoch model was selected as the final candidate, as it provides the best balance between performance and stability.

This experimentation phase confirms that extending training beyond the baseline configuration leads to measurable performance gains, while also demonstrating the effectiveness of transfer learning using pre-trained transformer models for domain-specific text classification tasks.